# 📈 Certificate Classification - Part 4: Deep Evaluation

## Goal
Deep dive into model performance, error analysis, and deployment readiness.

### Contents
1. ROC & Precision-Recall Curves
2. Threshold Optimization
3. Error Analysis
4. Cross-Validation
5. Deployment Considerations

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score,
    average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

In [ ]:
# Load data and model
data_dir = Path('./outputs/ml')

X = pd.read_pickle(data_dir / 'X_features.pkl')
y = pd.read_pickle(data_dir / 'y_target.pkl')

with open(data_dir / 'best_model.pkl', 'rb') as f:
    saved = pickle.load(f)
    best_name = saved['name']
    best_model = saved['model']
    scaler = saved['scaler']

print(f'Best model: {best_name}')
print(f'Features: {X.shape}')

In [ ]:
# Recreate train/test split (same seed)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale if needed
needs_scaling = 'Logistic' in best_name or 'Neural' in best_name or 'MLP' in best_name
if needs_scaling:
    X_test_eval = scaler.transform(X_test)
else:
    X_test_eval = X_test

# Get predictions
y_proba = best_model.predict_proba(X_test_eval)[:, 1]
y_pred = best_model.predict(X_test_eval)

## 2. ROC Curve

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'{best_name} (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Precision-Recall Curve

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color='darkorange', lw=2, label=f'{best_name} (AP = {avg_precision:.4f})')
ax.axhline(y=y_test.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test.mean():.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Threshold Optimization

In [ ]:
# Find optimal threshold for different objectives
thresholds = np.arange(0.1, 0.9, 0.05)
results = []

for thresh in thresholds:
    y_pred_t = (y_proba >= thresh).astype(int)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    results.append({'threshold': thresh, 'precision': prec, 'recall': rec, 'f1': f1})

thresh_df = pd.DataFrame(results)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='blue')
ax.plot(thresh_df['threshold'], thresh_df['recall'], label='Recall', color='green')
ax.plot(thresh_df['threshold'], thresh_df['f1'], label='F1 Score', color='red')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Metrics vs Classification Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Best thresholds
best_f1_idx = thresh_df['f1'].idxmax()
print(f"\n📌 Optimal threshold (max F1): {thresh_df.loc[best_f1_idx, 'threshold']:.2f}")
print(f"   Precision: {thresh_df.loc[best_f1_idx, 'precision']:.4f}")
print(f"   Recall: {thresh_df.loc[best_f1_idx, 'recall']:.4f}")
print(f"   F1: {thresh_df.loc[best_f1_idx, 'f1']:.4f}")

## 5. Confusion Matrix Analysis

In [ ]:
# Use optimal threshold
opt_thresh = thresh_df.loc[best_f1_idx, 'threshold']
y_pred_opt = (y_proba >= opt_thresh).astype(int)

cm = confusion_matrix(y_test, y_pred_opt)

print('Confusion Matrix (Optimal Threshold):')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')

print(f'\nFalse Positive Rate: {cm[0,1] / (cm[0,0] + cm[0,1]) * 100:.2f}%')
print(f'False Negative Rate: {cm[1,0] / (cm[1,0] + cm[1,1]) * 100:.2f}%')

## 6. Error Analysis

In [ ]:
# Analyze false positives and false negatives
test_df = X_test.copy()
test_df['y_true'] = y_test.values
test_df['y_pred'] = y_pred_opt
test_df['y_proba'] = y_proba

# False Positives (predicted malicious, actually benign)
fp = test_df[(test_df['y_true'] == 0) & (test_df['y_pred'] == 1)]
print(f'False Positives: {len(fp)}')

# False Negatives (predicted benign, actually malicious)
fn = test_df[(test_df['y_true'] == 1) & (test_df['y_pred'] == 0)]
print(f'False Negatives: {len(fn)}')

In [ ]:
# Analyze false negatives (missed malicious)
if len(fn) > 0:
    print('\n📊 False Negatives (Missed Malicious) Analysis:')
    print(f'  - Average probability: {fn["y_proba"].mean():.4f}')
    
    # Check which features differ
    tp = test_df[(test_df['y_true'] == 1) & (test_df['y_pred'] == 1)]
    if len(tp) > 0:
        print('\n  Feature comparison (FN vs TP):')
        for col in ['feat_sha1_sig', 'feat_weak_key', 'feat_cn_length', 'feat_cn_entropy']:
            if col in fn.columns:
                print(f'    {col}: FN={fn[col].mean():.3f} vs TP={tp[col].mean():.3f}')

## 7. Cross-Validation

In [ ]:
# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Prepare data (scale if needed)
if needs_scaling:
    X_cv = scaler.fit_transform(X)
else:
    X_cv = X.values if hasattr(X, 'values') else X

# Run CV
cv_scores = cross_val_score(best_model, X_cv, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print('5-Fold Cross-Validation (ROC-AUC):')
print(f'  Scores: {cv_scores}')
print(f'  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')

## 8. Deployment Recommendations

In [ ]:
print('='*60)
print('🚀 DEPLOYMENT RECOMMENDATIONS')
print('='*60)
print(f'''
MODEL: {best_name}

1. THRESHOLD SELECTION
   - Default (0.5): Balanced precision/recall
   - Optimal (F1): {opt_thresh:.2f}
   - High precision (fewer FP): Use 0.7+
   - High recall (catch more): Use 0.3-

2. PERFORMANCE EXPECTATIONS
   - ROC-AUC: {roc_auc:.4f}
   - Average Precision: {avg_precision:.4f}
   - CV Mean AUC: {cv_scores.mean():.4f}

3. LIMITATIONS
   - Class imbalance: {y.sum()} positive / {len(y)} total
   - Label noise: Domain blocklist may have false positives
   - Temporal drift: Retrain periodically

4. PRODUCTION CHECKLIST
   [ ] Feature pipeline (same engineering as training)
   [ ] Model serialization (pickle/joblib)
   [ ] Monitoring (prediction distribution drift)
   [ ] Feedback loop (collect new labels)
   [ ] A/B testing before full deployment

5. FILES TO DEPLOY
   - best_model.pkl (model + scaler + feature names)
   - feature_names.txt (feature order)
''')

In [ ]:
# Save evaluation results
output_dir = Path('./outputs/ml')

# Save threshold analysis
thresh_df.to_csv(output_dir / 'threshold_analysis.csv', index=False)

# Save curves data
pd.DataFrame({'fpr': fpr, 'tpr': tpr}).to_csv(output_dir / 'roc_curve.csv', index=False)
pd.DataFrame({'precision': precision, 'recall': recall}).to_csv(output_dir / 'pr_curve.csv', index=False)

print('✅ Saved evaluation results')